<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/week3/lesson11/Lesson11_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 11 — Exercises**
## Critical Points and Curvature: Valleys, Peaks and Flat Directions

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 3 — Calculus I\
**Lesson:** Lesson 11

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | Finding and classifying critical points | 10 |
| 2 | The second derivative | 15 |
| 3 | Taylor approximation and Newton's method | 15 |
| 4 | Convex and non-convex losses | 15 |
| 5 | Flat directions and ridge regularisation | 15 |
| 6 | Curvature and uncertainty for a tuning curve | 15 |
| 7 | Real data: is fitting a preferred direction convex? | 15 |
| **Total** | | **100** |
| Bonus | Sharp and flat minima | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def central_diff(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

def second_diff(f, x, h=1e-4):
    return (f(x + h) - 2 * f(x) + f(x - h)) / h**2

def hessian(f, p, rel=0.01):
    # Hessian of f(*p) by central second differences, step rel * |p_i| in parameter i (as in the teaching notebook).
    p = np.asarray(p, float); h = rel * np.abs(p); k = len(p); H = np.zeros((k, k))
    for a in range(k):
        for b in range(k):
            ea, eb = np.eye(k)[a] * h[a], np.eye(k)[b] * h[b]
            H[a, b] = (f(*(p + ea + eb)) - f(*(p + ea - eb)) - f(*(p - ea + eb)) + f(*(p - ea - eb))) / (4 * h[a] * h[b])
    return H

print("Setup complete.")

---
## Exercise 1 — Finding and Classifying Critical Points (10 points)

**1.1 (4 points, written response, no code).**

(a) A synaptic conductance after a spike is often modelled by the **alpha function** $g(t) = g_{\max}\,\frac{t}{\tau}\,e^{1 - t/\tau}$ for $t \ge 0$ ($t$ in ms, $\tau > 0$ in ms, $g$ in nS). Find its critical point(s), classify them with the first-derivative test, and give the largest value of $g$ on $[0, \infty)$.

(b) Find the global minimum and maximum of $f(x) = x^3 - 3x$ on the interval $[-3, 3]$. Where are they, and why does "set $f'(x) = 0$" not find them?

**Your answer:**

*(replace this line with your answer)*

In [ ]:
# 1.2 (4 points): Write find_critical(df, a, b, n=1000) that evaluates the derivative df on n equally spaced points in
# [a, b], finds each pair of neighbouring points where df changes sign, refines each by bisection to a tolerance of
# 1e-10, and returns a sorted NumPy array of the critical points. Apply it to f(x) = x^4 - 4x^3 + 4x^2 - 0.2x on
# [-1, 3] (write df1 by hand) and store the result in crit1. Then store the matching f'' values in d2_crit1 (use
# second_diff) and the types ('min' or 'max') in kind1.

def f1(x):
    return x**4 - 4 * x**3 + 4 * x**2 - 0.2 * x

# YOUR CODE HERE
crit1 = d2_crit1 = kind1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2
assert crit1 is not None, "Set crit1."
assert d2_crit1 is not None and kind1 is not None, "Set d2_crit1 and kind1."
assert len(crit1) == 3, "f1 has three critical points in [-1, 3]."
assert np.allclose(np.polyval([4, -12, 8, -0.2], crit1), 0, atol=1e-7), "These are not zeros of f1'."
assert list(kind1) == ['min', 'max', 'min'], "Check the signs of f'' at the critical points."
print("Self-check passed.")

**1.3 (2 points, written response).** Give two situations in which `find_critical` would miss a critical point, and say whether each matters when you are looking for minima.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 2 — The Second Derivative (15 points)

In [ ]:
# 2.1 (5 points): For f(x) = exp(x) at x = 1 (where f'' = e exactly), compute the absolute error of second_diff for
# h = np.logspace(-8, -1, 29). Store the errors in err2 (shape (29,)) and the h with the smallest error in h_best2.
# Plot the error against h on log-log axes.

h2 = np.logspace(-8, -1, 29)

# YOUR CODE HERE
err2 = h_best2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.1
assert err2 is not None and h_best2 is not None, "Set err2 and h_best2."
assert np.shape(err2) == (29,), "One error per h."
assert 1e-6 <= h_best2 <= 1e-2, "The best step should be far from both ends of the range."
assert err2[0] > 1e-3, "At h = 1e-8 rounding error should dominate."
print("Self-check passed.")

In [ ]:
# 2.2 (5 points): Write classify(f, x, h=1e-3, tol=1e-4) that returns 'min' if second_diff(f, x, h) > tol, 'max' if it
# is < -tol and 'inconclusive' otherwise. Apply it at the three critical points crit1 from Exercise 1 and at x = 0 for
# g(x) = x^4 and k(x) = -x^4; store the five results, in that order, in cls2.

# YOUR CODE HERE
cls2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.2
assert cls2 is not None, "Set cls2."
assert list(cls2) == ['min', 'max', 'min', 'inconclusive', 'inconclusive'], "Check classify; x^4 and -x^4 have f''(0) = 0."
print("Self-check passed.")

**2.3 (5 points, written response).** (a) The second-derivative test is inconclusive for both $x^4$ and $-x^4$ at 0. What does the first-derivative test say for each? (b) A neuron's response to stimulus concentration $c \ge 0$ follows the **Hill function** $r(c) = r_{\max}\,\frac{c^2}{c^2 + K^2}$. Find the concentration at which the response is most sensitive to small changes in $c$ (the inflection point), and compare it with the half-activation concentration $K$. What are the units of $r''(c)$?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 3 — Taylor Approximation and Newton's Method (15 points)

The **softplus** $f(x) = \ln(1 + e^x)$ is a smooth version of ReLU, used as an output nonlinearity for firing rates. Its derivatives are $f' = \sigma(x)$, $f'' = \sigma(1 - \sigma)$ and $f''' = \sigma(1 - \sigma)(1 - 2\sigma)$, with $\sigma$ the sigmoid.

In [ ]:
# 3.1 (5 points): At x0 = 0.5, compute the absolute errors of the Taylor approximations of orders 1, 2 and 3 to
# f(x0 + h) for h = np.logspace(-2, -0.5, 16). Fit a straight line to log(error) against log(h) for each order and store
# the three slopes in slopes3 (shape (3,)). Plot the errors on log-log axes.

softplus = lambda x: np.log1p(np.exp(x))
x0_3 = 0.5
h3 = np.logspace(-2, -0.5, 16)

# YOUR CODE HERE
slopes3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.1
assert slopes3 is not None, "Set slopes3."
assert np.shape(slopes3) == (3,), "One slope per order."
assert np.allclose(slopes3, [2, 3, 4], atol=0.25), "An order-n approximation should have error ~ h^(n+1)."
print("Self-check passed.")

In [ ]:
# 3.2 (6 points): A neuron's spike counts y (below) are modelled by a constant expected count exp(b). The Poisson loss is
# L(b) = mean(exp(b) - y * b). Write L'(b) and L''(b), then run Newton's method b <- b - L'(b) / L''(b) from b = 3 and
# from b = -3, stopping when |step| < 1e-10 (at most 100 steps). Store the two paths as lists in path_hi3 and path_lo3,
# and the exact minimiser in b_star3 (set L'(b) = 0 by hand).

y3 = np.random.default_rng(3).poisson(2.0, size=500)          # (500,) spike counts per 100 ms bin

# YOUR CODE HERE
path_hi3 = path_lo3 = b_star3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.2
assert path_hi3 is not None and path_lo3 is not None and b_star3 is not None, "Set path_hi3, path_lo3 and b_star3."
assert abs(b_star3 - np.log(y3.mean())) < 1e-9, "The minimiser solves exp(b) = mean(y)."
assert abs(path_hi3[-1] - b_star3) < 1e-8 and abs(path_lo3[-1] - b_star3) < 1e-8, "Both runs should end at b*."
assert len(path_lo3) > len(path_hi3), "Which start needs more steps, and why?"
print("Self-check passed.")

**3.3 (4 points, written response).** (a) Why does the run from $b = -3$ take so many more steps? Look at its first step. (b) Is $L(b)$ convex? Use $L''$. (c) In the teaching notebook, Newton's method on the double well went to the **maximum** from some starts. Why can that not happen here?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 4 — Convex and Non-Convex Losses (15 points)

Below are a stimulus $x$ and three kinds of response from 300 trials: a continuous rate, a spike count, and a binary "spiked or not". Each loss below depends on a single weight $w$.

In [ ]:
# 4.1 (6 points): Write convex_on_grid(f, ws) that returns True if every second difference f(w[i+1]) - 2 f(w[i]) + f(w[i-1])
# on the equally spaced grid ws is >= -1e-9 * max(1, max |f|), and False otherwise. Apply it on ws = np.linspace(-6, 6, 1201)
# to the four losses below and store the results in conv4 = {'a': ..., 'b': ..., 'c': ..., 'd': ...}.
#   (a) squared error      L(w) = mean((y_rate - w x)^2)
#   (b) Poisson loss       L(w) = mean(exp(w x) - y_count * w x)
#   (c) sigmoid squared    L(w) = mean((y_bin - sigmoid(w x))^2)
#   (d) absolute error     L(w) = mean(|y_rate - w x|)
# Plot the four losses (one panel each).

rng4 = np.random.default_rng(4)
x4 = rng4.normal(size=300)                                       # (300,) stimulus, standardised units
y_rate4 = 1.5 * x4 + rng4.normal(0, 1, 300)                      # (300,) rate, arbitrary units
y_count4 = rng4.poisson(np.exp(0.7 * x4))                        # (300,) spike counts
y_bin4 = (rng4.random(300) < sigmoid(2.0 * x4)).astype(float)    # (300,) 1 = spiked
ws4 = np.linspace(-6, 6, 1201)

# YOUR CODE HERE
conv4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.1
assert conv4 is not None, "Set conv4."
assert set(conv4) == {'a', 'b', 'c', 'd'}, "conv4 needs keys 'a' to 'd'."
assert conv4['a'] and conv4['b'] and conv4['d'], "Squared, Poisson and absolute-error losses of a linear predictor are convex."
assert not conv4['c'], "The sigmoid squared-error loss should fail the test somewhere in [-6, 6]."
print("Self-check passed.")

In [ ]:
# 4.2 (6 points): The non-convex loss below has several valleys. Run gradient descent w <- w - 0.01 * dLw(w) for 3000 steps
# from each of the 61 starts in np.linspace(-6, 6, 61). Store the end points in ends4 (shape (61,)), the distinct minima
# found (end points rounded to 2 decimals, sorted, unique) in minima4, and the fraction of starts that reach the global
# minimum in frac_global4.

Lw = lambda w: 0.05 * w**2 + np.sin(1.5 * w) + 0.1 * w           # a one-parameter loss with several valleys
dLw = lambda w: 0.1 * w + 1.5 * np.cos(1.5 * w) + 0.1

# YOUR CODE HERE
ends4 = minima4 = frac_global4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.2
assert ends4 is not None and minima4 is not None and frac_global4 is not None, "Set ends4, minima4 and frac_global4."
assert np.shape(ends4) == (61,), "One end point per start."
assert np.all(np.abs(dLw(ends4)) < 1e-3), "Gradient descent should end at points where the slope is (almost) zero."
assert len(minima4) >= 3, "This loss has at least three valleys in the starting range."
assert 0 < frac_global4 < 1, "Some, but not all, starts should reach the global minimum."
print("Self-check passed.")

**4.3 (3 points, written response).** (a) Explain why the sigmoid squared-error loss is not convex, while the squared error of a **linear** predictor is. (b) Show that $L(\mathbf{w}) = \frac1n\lVert\mathbf{y} - \mathbf{X}\mathbf{w}\rVert^2$ is convex for any data, using its Hessian $\frac2n\mathbf{X}^{\mathsf T}\mathbf{X}$. When is it strictly convex?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — Flat Directions and Ridge Regularisation (15 points)

A neuron's rate is modelled from two inputs, $u_1$ and $u_2$, with true weights $(2, 1)$ and Gaussian noise of SD 1. The inputs have correlation $\rho$. The loss is $L(\mathbf{w}) = \frac{1}{2n}\lVert\mathbf{y} - \mathbf{X}\mathbf{w}\rVert^2$, whose Hessian is $\mathbf{H} = \mathbf{X}^{\mathsf T}\mathbf{X}/n$ (as in the teaching notebook). The helper below builds $\mathbf{X}$.

In [ ]:
# PROVIDED -- do not modify.
def make_inputs(rho, n=200, seed=5):
    rng = np.random.default_rng(seed)
    u1 = rng.normal(size=n)
    u2 = rho * u1 + np.sqrt(1 - rho**2) * rng.normal(size=n)
    return np.column_stack([u1, u2])                              # (n, 2) inputs, standardised units
w_true5 = np.array([2.0, 1.0])
print("make_inputs is ready.")

In [ ]:
# 5.1 (4 points): For rho in [0, 0.9, 0.99, 0.999], build X = make_inputs(rho) and compute the eigenvalues of H = X.T @ X / n.
# Store the condition numbers (largest / smallest eigenvalue) in cond5 (shape (4,)) and, for rho = 0.99, the eigenvector
# of the smallest eigenvalue in v_flat5. Compare cond5 with (1 + rho) / (1 - rho).

rhos5 = [0.0, 0.9, 0.99, 0.999]

# YOUR CODE HERE
cond5 = v_flat5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.1
assert cond5 is not None and v_flat5 is not None, "Set cond5 and v_flat5."
assert np.shape(cond5) == (4,) and np.all(np.diff(cond5) > 0), "The condition number should grow with rho."
assert abs(abs(v_flat5[0] + v_flat5[1])) < 0.1, "The flat direction should be close to (1, -1)/sqrt(2), up to sign."
print("Self-check passed.")

In [ ]:
# 5.2 (5 points): Fix X = make_inputs(0.99). Using rng5 = np.random.default_rng(55), simulate 500 data sets
# y = X @ w_true5 + noise (noise SD 1) and fit least squares to each. Store the standard deviations of the estimates of
# w1 + w2 and w1 - w2 in sd_sum5 and sd_diff5. Also store the predictions of these SDs from curvature in pred5 = (pred_sum,
# pred_diff), using Cov(w_hat) = sigma^2 (X^T X)^{-1} with sigma = 1 and Var(v^T w_hat) = v^T Cov v.

X5 = make_inputs(0.99)
rng5 = np.random.default_rng(55)

# YOUR CODE HERE
W_hat5 = sd_sum5 = sd_diff5 = pred5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.2
assert sd_sum5 is not None and sd_diff5 is not None and pred5 is not None, "Set sd_sum5, sd_diff5 and pred5."
assert sd_diff5 > 5 * sd_sum5, "The difference w1 - w2 should be far less well determined than the sum."
assert abs(sd_sum5 / pred5[0] - 1) < 0.15 and abs(sd_diff5 / pred5[1] - 1) < 0.15, "Simulation and curvature should agree within ~15%."
print("Self-check passed.")

In [ ]:
# 5.3 (4 points): Ridge regression: w_lam = (H + lam I)^{-1} X^T y / n with H = X^T X / n. For each lam in
# np.logspace(-4, 1, 21), fit all 500 simulated data sets of 5.2 and store the mean of ||w_lam - w_true5||^2 in mse_w5
# (shape (21,)) and the best lam in lam_best5. Plot mse_w5 against lam (log x-axis).

lams5 = np.logspace(-4, 1, 21)

# YOUR CODE HERE
mse_w5 = lam_best5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.3
assert mse_w5 is not None and lam_best5 is not None, "Set mse_w5 and lam_best5."
assert np.shape(mse_w5) == (21,), "One value per lambda."
assert lam_best5 > 1e-3 and mse_w5.min() < 0.7 * mse_w5[0], "A moderate penalty should beat (almost) no penalty clearly."
assert mse_w5[-1] > mse_w5.min(), "Too large a penalty should hurt again."
print("Self-check passed.")

**5.4 (2 points, written response).** Explain the shape of the `mse_w5` curve in terms of curvature, bias and variance. What would a neuroscientist wrongly conclude from a single least-squares fit with $\rho = 0.99$?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 6 — Curvature and Uncertainty for a Tuning Curve (15 points)

A visual neuron's response to stimulus orientation $\theta$ (degrees) is modelled by a Gaussian tuning curve

$$r(\theta) = b + a\,\exp\!\Big(-\frac{(\theta - \mu)^2}{2s^2}\Big),$$

with baseline $b$ and amplitude $a$ in spikes/s, preferred orientation $\mu$ and width $s$ in degrees. The simulated data below have true values $b = 10$, $a = 30$, $\mu = 10$, $s = 25$, and independent Gaussian noise of SD 5 spikes/s.

In [ ]:
# 6.1 (6 points): Fit the model with curve_fit (p0 = [5, 20, 0, 20]). Compute the Hessian of the mean squared error at the
# fit with hessian(), and from it the curvature-based covariance 2 sigma2_hat / n * H^{-1} (sigma2_hat = the mean squared
# residual). Store the fitted parameters in p6, the standard errors in se_curv6 (shape (4,)) and the correlation matrix of
# the estimates in corr6 (shape (4, 4)).

theta6 = np.linspace(-90, 90, 400)                                # (400,) orientations, degrees
def tuning(th, b, a, mu, s):
    return b + a * np.exp(-(th - mu)**2 / (2 * s**2))
p_true6 = np.array([10.0, 30.0, 10.0, 25.0])
y6 = tuning(theta6, *p_true6) + np.random.default_rng(6).normal(0, 5, theta6.size)   # (400,) spikes/s

# YOUR CODE HERE
p6 = se_curv6 = corr6 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.1
assert p6 is not None and se_curv6 is not None and corr6 is not None, "Set p6, se_curv6 and corr6."
assert np.shape(se_curv6) == (4,) and np.shape(corr6) == (4, 4), "Check the shapes."
assert np.allclose(np.diag(corr6), 1, atol=1e-6), "A correlation matrix has ones on its diagonal."
assert np.all(np.abs(p6 - p_true6) < 4 * se_curv6 + 1e-9), "The fit should be within a few standard errors of the truth."
print("Self-check passed.")

In [ ]:
# 6.2 (6 points): Check the curvature-based standard errors by simulation: with rng6 = np.random.default_rng(66), draw 300
# new noise vectors (SD 5), refit each simulated data set with curve_fit (p0 = p_true6), and store the standard deviations
# of the 300 estimates of each parameter in se_sim6 (shape (4,)) and the correlation of the b and s estimates across the
# simulations in corr_bs6.

rng6 = np.random.default_rng(66)

# YOUR CODE HERE
P_sim6 = se_sim6 = corr_bs6 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.2
assert se_sim6 is not None and corr_bs6 is not None, "Set se_sim6 and corr_bs6."
assert np.shape(se_sim6) == (4,), "One SD per parameter."
assert np.all(np.abs(se_sim6 / se_curv6 - 1) < 0.25), "With independent noise, simulation and curvature should agree within ~25%."
assert np.sign(corr_bs6) == np.sign(corr6[0, 3]), "Simulation and curvature should agree on the sign of the b-s correlation."
print("Self-check passed.")

**6.3 (3 points, written response).** (a) Which pair of parameters trades off most strongly, and why does that make sense for this curve? (b) Here curvature predicted the uncertainty well; in the teaching notebook's mini-lab it underestimated the uncertainty of $r_0$ and $A$ by a factor of 2–3. What is different about the two situations?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: Is Fitting a Preferred Direction Convex? (15 points)

Motor-cortex neurons are tuned to the direction of hand movement (Lessons 2 and 4). Here you fit a **von Mises** tuning curve to every neuron of the motor-cortex data set ([Stevenson et al., 2011](https://doi.org/10.1152/jn.00626.2010); [DREAM on CRCNS](https://crcns.org/data-sets/movements/dream); packaged by [Benjamin et al., 2018](https://github.com/KordingLab/spykesML)):

$$r(\theta) = b + a\,e^{\kappa(\cos(\theta - \theta_p) - 1)},\qquad a \ge 0,$$

where $\theta$ is the movement direction 100 ms after the rate bin, $\theta_p$ the preferred direction, and $\kappa$ sets the width of the bump (larger $\kappa$, narrower bump). For fixed $\kappa$ and $\theta_p$ the model is linear in $(b, a)$, so the best $(b, a)$ follow from least squares, and the loss becomes a function of $\theta_p$ alone: a **profile**. If the best $a$ is negative (a dip instead of a bump), the constraint $a \ge 0$ makes the best fit flat ($a = 0$, $b = $ mean rate), so the loss equals the variance of the rate.

The cell below keeps the training half and the bins in which the hand is moving (speed above the training median), and provides the grid of preferred directions. Do not modify it.

In [ ]:
# PROVIDED -- do not modify.
import os, urllib.request
from scipy.io import loadmat

M1_FILE = 'm1_stevenson_2011.mat'
if not os.path.exists(M1_FILE):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/KordingLab/spykesML/ff2b1830ae07909e5f1a0811445ca3fd5732dc6f/data/' + M1_FILE, M1_FILE)
m1 = loadmat(M1_FILE)
rates = m1['spikes'].T.astype(float) / 0.05                      # (15536, 171) spikes/s
vel = m1['handVel'][:2].T * 100                                  # (15536, 2) cm/s
Y7, V7 = rates[:-2], vel[2:]                                     # rate bins and the velocity 100 ms later
speed7 = np.linalg.norm(V7, axis=1)
train7 = np.arange(len(Y7)) < len(Y7) // 2
moving7 = train7 & (speed7 > np.median(speed7[train7]))
theta7 = np.arctan2(V7[moving7, 1], V7[moving7, 0])              # (3883,) movement direction, radians
R7 = Y7[moving7]                                                 # (3883, 171) rates in those bins, spikes/s
neurons7 = np.where(R7.std(axis=0) > 0)[0]                       # the 167 neurons that fire in these bins
grid7 = np.radians(np.arange(0, 360, 2))                         # (180,) candidate preferred directions
kappas7 = [1, 2, 4, 8]
print("bins:", R7.shape, " neurons:", neurons7.size, " grid:", grid7.size)

In [ ]:
# 7.1 (5 points): Write profile7(y, kappa, tp) returning the mean squared error of the best fit with a >= 0 (the variance of
# y if the least-squares a is negative). For every neuron in neurons7 and every kappa in kappas7, compute the profile on
# grid7 and count its local minima ON THE CIRCLE (a grid point lower than both neighbours, with the first and last points
# neighbours). Store the counts in nmin7 = {kappa: integer array of shape (167,)}, the fraction of variance explained at the
# best grid point, 1 - min(profile) / var(y), in r2_7 = {kappa: array (167,)}, and the best preferred directions (radians)
# in tp7 = {kappa: array (167,)}. Print, for each kappa, how many neurons have exactly one local minimum.

# YOUR CODE HERE
nmin7 = r2_7 = tp7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.1
assert nmin7 is not None and r2_7 is not None and tp7 is not None, "Set nmin7, r2_7 and tp7."
assert set(nmin7) == set(kappas7) and all(np.shape(v) == (neurons7.size,) for v in nmin7.values()), "One count per neuron per kappa."
assert all(np.all(v >= 1) for v in nmin7.values()), "Every profile on a circle has at least one local minimum."
assert np.sum(nmin7[1] == 1) > np.sum(nmin7[8] == 1), "Narrower bumps (larger kappa) should give more neurons with several minima."
assert np.all((r2_7[1] >= 0) & (r2_7[1] < 1)), "r2_7 is a fraction of variance."
print("Self-check passed.")

In [ ]:
# 7.2 (3 points): Group the neurons by how well tuned they are at that kappa (r2_7 below 0.01, 0.01 to 0.05, above 0.05) and
# store the fraction of neurons in each group with more than one local minimum in frac7, an array of shape (2, 3) with rows
# kappa = 1 and kappa = 8. Print it with the group sizes.

# YOUR CODE HERE
frac7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.2
assert frac7 is not None and np.shape(frac7) == (2, 3), "frac7 should be 2 x 3."
assert frac7[1, 0] > frac7[1, 2], "At kappa = 8, poorly tuned neurons should have several minima more often than well-tuned ones."
print("Self-check passed.")

In [ ]:
# 7.3 (3 points): For neuron 161 at kappa = 8, plot its profile over the preferred direction (degrees) and mark the local
# minima. Then run a DISCRETE downhill walk on the grid from each of the 12 starts at 0, 30, ..., 330 degrees: repeatedly
# move to the lower of the two neighbouring grid points while it is strictly lower than the current point. Store the 12 end
# points in degrees in ends7 and the number of walks that end on a plateau (where the profile equals var(y), the a = 0 fit)
# in n_plateau7.

# YOUR CODE HERE
ends7 = n_plateau7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.3
assert ends7 is not None and n_plateau7 is not None, "Set ends7 and n_plateau7."
assert len(ends7) == 12, "One end point per start."
assert len(set(np.round(ends7).astype(int))) >= 3, "Walks from different starts should end in different places."
assert n_plateau7 >= 1, "At least one walk should stop on the flat a = 0 plateau."
print("Self-check passed.")

In [ ]:
# 7.4 (2 points): The convex alternative. Fit r = c0 + c1 cos(theta) + c2 sin(theta) to every neuron by ordinary least
# squares (ONE linear solve for all neurons), take tp_lin7 = arctan2(c2, c1), and store the median absolute angular
# difference (degrees, wrapped to [0, 180]) between tp_lin7 and tp7[1] in dtheta7_all, and the same median over neurons
# with r2_7[1] > 0.05 in dtheta7_tuned.

# YOUR CODE HERE
tp_lin7 = dtheta7_all = dtheta7_tuned = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.4
assert tp_lin7 is not None and dtheta7_all is not None and dtheta7_tuned is not None, "Set tp_lin7, dtheta7_all and dtheta7_tuned."
assert np.shape(tp_lin7) == (neurons7.size,), "One preferred direction per neuron."
assert 0 <= dtheta7_tuned < 10, "For well-tuned neurons the two methods should agree within a few degrees."
print("Self-check passed.")

**7.5 (2 points, written response — interpretation and limitation).** Using your results, answer: is fitting a preferred direction a convex problem? When do multiple minima appear, and when do they matter? Why does the cos/sin fit avoid the problem, and what does it give up? Give one limitation of this analysis.

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — Sharp and Flat Minima (extra credit, not part of the 100 points)

Machine-learning folklore says that **flat** minima generalise better than **sharp** ones, because a small change in the data (or the weights) changes the loss of a flat minimum less.

1. Take one simulated data set from Exercise 5 ($\rho = 0.99$; the first row of `Ysim5`) and its least-squares fit. Perturb the fitted weights by a step of length $\varepsilon = 0.5$ in 200 random directions and record how much the training loss rises. Compare each rise with the quadratic prediction $\frac12\varepsilon^2\,\mathbf{v}^{\mathsf T}\mathbf{H}\mathbf{v}$.
2. Which directions are cheapest, and which most expensive? Relate the range of rises to the two eigenvalues of $\mathbf{H}$.
3. A flat minimum can be made sharp just by rescaling a parameter (measuring a weight in different units). What does this imply for "flatness" as a measure of generalisation?

**Your code:**

In [ ]:
# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, and a legend where appropriate.
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.